### Understanding Transformers

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')

In [5]:
import math
import sys, os

In [6]:
import torch
import torch.nn as nn
from torchtext.datasets import Multi30k

### Step-0:- Load the Dataset
The dataset is used for language translation purposes

In [7]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# ! wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [8]:

from datasets import load_dataset
from pathlib import Path

DATASET_NAME = "opus_books" 

In [9]:
ds = load_dataset(DATASET_NAME, "en-it", split="train")
len(ds), type(ds), ds[3]

(32332,
 datasets.arrow_dataset.Dataset,
 {'id': '3', 'translation': {'en': 'CHAPTER I', 'it': 'PARTE PRIMA'}})

In [10]:
import random
examples = random.sample(range(len(ds)), 3)
src_text = ds[examples[0]]["translation"]["en"]
tgt_text = ds[examples[0]]["translation"]["it"]
print("One Example:")
print("English:", src_text)
print("Italian:", tgt_text)
print("-----------")
for i in examples:
    print(ds[i])
    print("---")

One Example:
English: There was a pause, and the mother exchanged glances with her daughter.
Italian: Seguì un silenzio. La madre e la figlia si guardarono ancora una volta.
-----------
{'id': '29450', 'translation': {'en': 'There was a pause, and the mother exchanged glances with her daughter.', 'it': 'Seguì un silenzio. La madre e la figlia si guardarono ancora una volta.'}}
---
{'id': '13180', 'translation': {'en': 'It is the most fairy-like little nook on the whole river.', 'it': 'È il più incantevole cantuccio del Tamigi.'}}
---
{'id': '19654', 'translation': {'en': "'Oh, yes, you saved that Levin from unpleasantness.'", 'it': '— Come! Avete salvato quel Levin da un incidente increscioso.'}}
---


In [11]:
from src.config import get_config
config = get_config()
config

{'batch_size': 8,
 'num_epochs': 20,
 'lr': 0.0001,
 'seq_len': 350,
 'd_model': 512,
 'datasource': 'opus_books',
 'lang_src': 'en',
 'lang_tgt': 'it',
 'model_folder': 'weights',
 'model_basename': 'tmodel_',
 'preload': 'latest',
 'tokenizer_file': 'tokenizer_{0}.json',
 'experiment_name': 'runs/tmodel'}

### Step-1:- Load Dataset and Tokenize it

In [12]:
from src.dataset_path import DATASET_DIR, complete_tokenization

In [13]:
train_dataloader,val_dataloader, tokenizer_src, tokenizer_tgt = complete_tokenization(config, ds_raw=None)

Max Length of Source Sentence: 309
Max Length of Target Sentence: 274


In [14]:
sample_batch = next(iter(val_dataloader))
sample_batch.keys()

dict_keys(['encoder_input', 'decoder_input', 'encoder_mask', 'decoder_mask', 'label', 'src_text', 'tgt_text'])

In [15]:
### Our requirement is Dimension size and vocab size
DIMENSION_SIZE = 512

### Step-2:- Focus on the Input Embeddings first 
This is where we convert text to numbers. Now, first we convert text to embeddings(not tokens). Assume input is tokens not raw text


This is the first part, It takes the input and coverts it into usable embeddings(text to numbers). <br>
First we assign each text a token( a number in vocabulary), then we use an embedding model to create an embedding.



Since this is the input to the model, we need to figure out certain things first :- 

1. We need to know input size of the text model, d_model is the input size to the model(same reference as the paper). It is the embedding size(how big the embedding shape is going to be)
2. vocab_size: - How many words are there in the vocabulary (total unique words)
    

In [16]:
from src.input_embd import InputEmbedding

In [17]:
encoder_input = InputEmbedding(dimension_size=DIMENSION_SIZE, vocab_size=tokenizer_src.get_vocab_size())
encoder_input

InputEmbedding(
  (embedding_layer): Embedding(15698, 512)
)

In [53]:
print(sample_batch['encoder_input'].shape)
x = encoder_input(sample_batch['encoder_input'])
print(x.shape)

torch.Size([1, 350])
torch.Size([1, 350, 512])


In [58]:
print("Token ID:", sample_batch['encoder_input'][0, 1])
print("Token:", tokenizer_src.id_to_token(sample_batch['encoder_input'][0, 1].item()))
print("Embedding:", x.shape)
print("Embedding shape:", x[0, 0].shape)

Token ID: tensor(12)
Token: '
Embedding: torch.Size([1, 350, 512])
Embedding shape: torch.Size([512])


 #### Step-2: - Positional Encoding
Since we have converted now text to embeddings, we want to add some position information, that tells the model the information about the position of the word in a sentence. 

In [59]:
### Here we need to figure out what it is the context length
from src.input_embd import PositionalEncoding

In [60]:
position_encoder= PositionalEncoding(dimension_size=DIMENSION_SIZE, max_length=config["seq_len"], dropout=0.0)   # use 0 for inspection)
x = position_encoder(x)

print("Before PE:", x.shape)
print("After PE:", x.shape)
print(position_encoder.pc_enc_matrix.shape)

Before PE: torch.Size([1, 350, 512])
After PE: torch.Size([1, 350, 512])
torch.Size([1, 350, 512])


In [61]:
# token_position_1_before = embd[0, 1, :10]
# token_position_1_after = pos_embd[0, 1, :10]

# print("Before:")
# print(token_position_1_before)

# print("Position encoding:")
# print(position_encoder.pc_enc_matrix[0, 1, :10])

# print("After:")
# print(token_position_1_after)

#### Step-3: - Understanding Self Attention

Trying to understand self attention Properly


In [52]:
from src.attention import SingleHeadAttention

In [63]:
# 1. Test your SingleHeadAttention once
head = SingleHeadAttention(d_in=DIMENSION_SIZE, d_out=64, dropout=0.0)
out = head(x)

In [65]:
print(out)
print(out.out.shape)
print(out.weights.shape)
print(out.weights[0].sum(dim=-1))

AttentionOutput(out=tensor([[[  5.9166,  -1.0431,   4.7400,  ...,  16.5445, -14.2525,  -6.4088],
         [ -8.6399,  -9.2433,  -0.8038,  ..., -10.9833, -12.9781, -16.1502],
         [ -2.0776,   5.0746,  14.0846,  ...,   1.3472,  22.1937,  -5.2413],
         ...,
         [  5.9166,  -1.0431,   4.7400,  ...,  16.5445, -14.2525,  -6.4088],
         [  5.9166,  -1.0431,   4.7400,  ...,  16.5445, -14.2525,  -6.4088],
         [  5.9166,  -1.0431,   4.7400,  ...,  16.5445, -14.2525,  -6.4088]]],
       grad_fn=<UnsafeViewBackward0>), weights=tensor([[[0.0000e+00, 0.0000e+00, 1.5204e-24,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 5.0951e-42, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         ...,
         [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00,
          0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00,

#### Step-4: - Layer Normalization

It will makes sure that batches are separate, that means the normalization for that batch ensures the mean and std of the batch remains separate
